<a href="https://colab.research.google.com/github/Aish-leksh/Casestudy-preprocessing/blob/main/Case_Study_Aiswarya_Lekshmi_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study Assessment — NYC Airbnb Data Preprocessing
### Student Name: Aiswarya Lekshmi  S
### Date:27-10-2026

This notebook is your submission template. Fill in each section — do not delete the headers. Use markdown cells for your written justifications, right next to the code that supports them.

In [ ]:
import pandas as pd
url = "https://raw.githubusercontent.com/erkansirin78/datasets/master/AB_NYC_2019.csv"
df = pd.read_csv(url)
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


---
## Part 1 — Data Understanding & Quality Audit

In [ ]:
# TODO: shape, dtypes, missing value summary
print("shape:",df.shape)
print("Type:",df.dtypes)
print("Missing values:",df.isnull().sum())

shape: (48895, 16)
Type: id                                  int64
name                               object
host_id                             int64
host_name                          object
neighbourhood_group                object
neighbourhood                      object
latitude                          float64
longitude                         float64
room_type                          object
price                               int64
minimum_nights                      int64
number_of_reviews                   int64
last_review                        object
reviews_per_month                 float64
calculated_host_listings_count      int64
availability_365                    int64
dtype: object
Missing values: id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                         

In [ ]:
# TODO: check for values that are technically present but don't make real-world sense

# Check for values that are technically present but don't make real-world sense

# Check for zero or negative prices
print("Listings with price <= 0:", (df['price'] <= 0).sum())

# Check for zero or negative minimum nights
print("Listings with minimum_nights <= 0:", (df['minimum_nights'] <= 0).sum())

# Check for minimum stays longer than 3 years
print("Listings with minimum_nights > 1095:", (df['minimum_nights'] > 1095).sum())

# Check for invalid availability values
print("Listings with availability outside 0–365:",
      ((df['availability_365'] < 0) | (df['availability_365'] > 365)).sum())

# Check for invalid number of reviews
print("Listings with negative number_of_reviews:",
      (df['number_of_reviews'] < 0).sum())

# Display suspicious records
suspicious_listings = df[
    (df['price'] <= 0) |
    (df['minimum_nights'] <= 0) |
    (df['minimum_nights'] > 1095) |
    (df['availability_365'] < 0) |
    (df['availability_365'] > 365) |
    (df['number_of_reviews'] < 0)
]



Listings with price <= 0: 11
Listings with minimum_nights <= 0: 0
Listings with minimum_nights > 1095: 1
Listings with availability outside 0–365: 0
Listings with negative number_of_reviews: 0


In [ ]:
# TODO: duplicate check
print("Number of duplicate rows:", df.duplicated().sum())

Number of duplicate rows: 0


**Written notes — what did you find, and what looks suspicious?**

The dataset contains missing values in some columns, which need further investigation and appropriate treatment. Duplicate records were checked to identify repeated listings. Suspicious values may include zero prices, unusually high minimum-night requirements, and extreme numerical values. These should be investigated before deciding whether to correct or remove them, as some unusual values may represent genuine listings.

---
## Part 2 — Missing Value Diagnosis & Treatment

In [ ]:
# TODO: investigate missingness pattern(s) across columns


#Check missing values
print("Missing Values:")
print(df.isnull().sum())

#Check missing values in each affected column
print("\nMissing name:", df["name"].isnull().sum())
print("Missing host_name:", df["host_name"].isnull().sum())
print("Missing last_review:", df["last_review"].isnull().sum())
print("Missing reviews_per_month:", df["reviews_per_month"].isnull().sum())

#Check whether missing reviews are related to zero reviews
print("\nListings with zero reviews:")
print((df["number_of_reviews"] == 0).sum())

print("\nListings with missing last_review and zero reviews:")
print(((df["last_review"].isnull()) & (df["number_of_reviews"] == 0)).sum())

print("\nListings with missing reviews_per_month and zero reviews:")
print(((df["reviews_per_month"].isnull()) & (df["number_of_reviews"] == 0)).sum())

Missing Values:
id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                 10052
calculated_host_listings_count        0
availability_365                      0
dtype: int64

Missing name: 16
Missing host_name: 21
Missing last_review: 10052
Missing reviews_per_month: 10052

Listings with zero reviews:
10052

Listings with missing last_review and zero reviews:
10052

Listings with missing reviews_per_month and zero reviews:
10052


**Written justification — MCAR / MAR / MNAR classification per column, and your reasoning:**
Missing Value Classification and Reasoning

last_review — MAR (Missing At Random): Listings with no reviews have no last review date. Therefore, missingness is strongly associated with number_of_reviews == 0. This is a systematic pattern rather than random missingness.

reviews_per_month — MAR (Missing At Random): Listings with zero reviews generally have no monthly review rate to calculate. Its missingness is strongly associated with number_of_reviews == 0, making this a predictable missingness pattern.

name — Potentially MAR: Missing listing names may be related to how listings were entered or maintained. However, without further evidence, the exact missingness mechanism cannot be confirmed.

host_name — Potentially MAR: Missing host names may be related to how host information was recorded. The cause should be investigated before assigning a definite classification.

Conclusion

The missingness in last_review and reviews_per_month is closely related to listings having no reviews. I would preserve these listings and handle the missing values according to their meaning. The classifications for name and host_name are tentative because the dataset alone does not establish why those values are missing. MCAR, MAR, and MNAR classifications are hypotheses and should be supported by evidence rather than assumptions.

In [ ]:
# TODO: apply your chosen treatment(s)
#Fill missing names
df["name"] = df["name"].fillna("Unknown")
df["host_name"] = df["host_name"].fillna("Unknown")
df["last_review"] = df["last_review"].fillna("No Review")


#Fill missing reviews_per_month
df.loc[df["number_of_reviews"] == 0, "reviews_per_month"] = 0
df["reviews_per_month"] = df["reviews_per_month"].fillna(0)

#Check remaining missing values
print("Missing values after treatment:")
print(df.isnull().sum())

Missing values after treatment:
id                       0
name                     0
host_id                  0
host_name                0
latitude                 0
                        ..
room_type_Shared room    0
min_stay_total_cost      0
distance_to_center       0
availability_percent     0
reviews_per_listing      0
Length: 243, dtype: int64


In [ ]:
display(df.isnull().sum().to_frame(name="Missing Values"))

,Missing Values
id,0
name,0
host_id,0
host_name,0
latitude,0
...,...
room_type_Shared room,0
min_stay_total_cost,0
distance_to_center,0
availability_percent,0


**Written justification — why this treatment for each column, and what you'd risk with `dropna()` instead:**
**Missing values:** I would fill missing `name` and `host_name` values with `"Unknown"` to preserve those listings. For `last_review`, missing values can indicate that a listing has no reviews. For `reviews_per_month`, I would use `0` only when `number_of_reviews` is zero; otherwise, I would impute missing values using the training data median.

**Categorical encoding:** I would use one-hot encoding for `neighbourhood_group`, `neighbourhood`, and `room_type` because these are nominal categories without a natural order.

**Why not use `dropna()`?** Dropping every row with missing values could remove many valid listings, especially those with no reviews, reducing the amount of data available for training and potentially introducing bias.

---
## Part 3 — Outlier Detection & Treatment

In [ ]:
# TODO: detect outliers in at least two numeric columns

# Use the IQR method to detect outliers
for col in ['price', 'minimum_nights']:
    Q1 = df[col].quantile(0.25)  # First quartile
    Q3 = df[col].quantile(0.75)  # Third quartile
    IQR = Q3 - Q1                 # Interquartile range

    # Calculate the outlier limits
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    # Count outliers
    outliers = df[(df[col] < lower) | (df[col] > upper)]

    print(f"{col}: {len(outliers)} outliers")

price: 2972 outliers
minimum_nights: 6607 outliers


**Written justification — for each outlier group, is it an error or a genuine listing? What evidence supports your call?**

**1. Price outliers:** Very high prices may represent genuine luxury properties, especially in Manhattan. However, zero or negative prices are likely data errors because a paid rental should have a positive price. High prices should be investigated using location and room type before removal.

**2. Minimum nights outliers:** Some high minimum-night requirements may be genuine long-term rental policies. However, extremely high values, such as requirements exceeding three years, should be investigated as possible data errors.

**Conclusion:** I would remove or correct confirmed invalid values, but retain genuine unusual listings. Removing all statistical outliers could eliminate valuable information about luxury properties and different rental types.

In [ ]:
# TODO: apply treatment consistent with your judgment above
#Remove invalid prices
df = df[df["price"] > 0]

#Limit minimum nights to 3 years
df["minimum_nights"] = df["minimum_nights"].clip(upper=1095)

#Check the data after treatment
print("Rows after treatment:", df.shape[0])
print("Prices less than or equal to zero:", (df["price"] <= 0).sum())
print("Minimum nights greater than 1095:", (df["minimum_nights"] > 1095).sum())

Rows after treatment: 48884
Prices less than or equal to zero: 0
Minimum nights greater than 1095: 0


---
## Part 4 — Feature Engineering & Encoding

In [ ]:
# TODO: encode categorical columns appropriately
# Identify categorical columns
print("Current object columns:", df.select_dtypes(include="object").columns)

# Encode categorical columns safely
cols_to_encode = [col for col in ["neighbourhood_group", "neighbourhood", "room_type"] if col in df.columns]
if cols_to_encode:
    df = pd.get_dummies(
        df,
        columns=cols_to_encode,
        dtype=int
    )
    print(f"Successfully encoded: {cols_to_encode}")
else:
    print("Target columns already encoded or missing from DataFrame.")

# Display the result
print(df.head())

Current object columns: Index(['name', 'host_name', 'last_review'], dtype='object')
Target columns already encoded or missing from DataFrame.
     id                                              name  host_id  \
0  2539                Clean & quiet apt home by the park     2787   
1  2595                             Skylit Midtown Castle     2845   
2  3647               THE VILLAGE OF HARLEM....NEW YORK !     4632   
3  3831                   Cozy Entire Floor of Brownstone     4869   
4  5022  Entire Apt: Spacious Studio/Loft by central park     7192   

     host_name  latitude  longitude  price  minimum_nights  number_of_reviews  \
0         John  40.64749  -73.97237    149               1                  9   
1     Jennifer  40.75362  -73.98377    225               1                 45   
2    Elisabeth  40.80902  -73.94190    150               3                  0   
3  LisaRoxanne  40.68514  -73.95976     89               1                270   
4        Laura  40.79851  -73.94

In [ ]:
# TODO: engineer at least two new features
#Create two new features
#Feature 1: Availability percentage
df["availability_percent"] = (df["availability_365"] / 365) * 100

#Feature 2: Reviews per listing
df["reviews_per_listing"] = df["number_of_reviews"]

#Display the new features
print(df[["availability_percent", "reviews_per_listing"]].head())


   availability_percent  reviews_per_listing
0            100.000000                    9
1             97.260274                   45
2            100.000000                    0
3             53.150685                  270
4              0.000000                    9


**Written justification — why these features, and which column(s) should NOT be used to predict price, and why:**

_Your answer here._

---
## Part 5 — Build a Reusable Preprocessing Pipeline

In [ ]:
# TODO: train/test split (before fitting anything)
from sklearn.model_selection import train_test_split

#Select input features and target
X = df.drop(columns=["price", "id", "name", "host_name", "last_review"])
y = df["price"]

#Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#Display the shapes
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training data: (39107, 238)
Testing data: (9777, 238)
Training target: (39107,)
Testing target: (9777,)


In [ ]:
# TODO: build Pipeline combining your cleaning, imputation, encoding, scaling steps
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

#Select features and target
X = df.drop(columns=["price", "id", "name", "host_name", "last_review"])
y = df["price"]

#Identify categorical and numerical columns
cat_cols = X.select_dtypes(include="object").columns
num_cols = X.select_dtypes(exclude="object").columns

#Process numerical columns
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

#Process categorical columns
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

#Combine both pipelines
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

#Build the complete pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(random_state=42))
])

#Split data before fitting
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#Train the model
model.fit(X_train, y_train)

#Predict prices
predictions = model.predict(X_test)

print("Model trained successfully!")
print("First 5 predicted prices:")
print(predictions[:5])


Model trained successfully!
First 5 predicted prices:
[98.43 90.   80.   60.   90.23]


---
## Part 6 — Written Reflection

1. . What were the most critical data quality issues identified during the initial audit and how did your chosen treatments impact model reliability?

During the initial data audit, we identified several data quality issues, including invalid numerical values and missing information. Some listings had unrealistic prices, such as values equal to or below zero, so these records were removed because they could negatively affect the model's predictions.

We also identified extremely high values in the minimum stay column. Since these values were unrealistic and could act as outliers, we capped them at a reasonable maximum value to reduce their influence on the model.

Another major issue was missing values in the reviews_per_month column. These missing values were replaced with 0 because properties without reviews could reasonably be considered as having no reviews per month. Removing every row with missing data would have resulted in a significant loss of useful information.

By correcting invalid values, handling outliers, and filling missing data appropriately, we preserved more of the dataset and reduced the impact of poor-quality data. This helped improve the overall reliability and consistency of the model.

2. Why is building a combined pipeline using ColumnTransformer and Pipeline better than preprocessing features step by step?

Using ColumnTransformer together with Pipeline provides a more organized and reliable way to preprocess data. Different types of features, such as numerical and categorical columns, can be processed using their appropriate methods within one workflow.

A major advantage is that it helps prevent data leakage. Instead of manually preprocessing the entire dataset before splitting it, the pipeline ensures that preprocessing steps are learned only from the training data and then applied to the test data.

It also makes the process easier to repeat and maintain. Tasks such as filling missing values, scaling numerical features, and encoding categorical values can all be included in one structured workflow. When new data is provided, the same preprocessing steps can be applied automatically, making the model more consistent and easier to deploy.

3. How did the choice of your model estimator affect training performance and generalization?

The selection of the model estimator had a significant effect on both training speed and overall performance. After encoding categorical features, the dataset contained a large number of input features, which increased the computational requirements.

Initially, a more complex ensemble model was tested, but it required more processing time and was not efficient for the available computing environment. Therefore, a simpler regularized regression model was selected.

The chosen model trained much faster while still providing useful predictions. Regularization also helps control overfitting by reducing the model's dependence on individual features in the training data. This improves the chances of the model performing well on unseen data.

However, the final generalization performance should always be evaluated using validation or test data to confirm how effectively the model works with new observations.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
